## 登陆下载数据集InternVid

In [ ]:

!huggingface-cli login



    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To login, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: fineGrained).
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your termin

In [ ]:
!pip install datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 10.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 15.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 16.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 23.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 17.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.31.0
    Uninstalling requests-2.31.0:
      Successfully uninstalled requests-2.31.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 

In [ ]:
from datasets import load_dataset
ds = load_dataset("OpenGVLab/InternVid", "InternVid-10M")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating FLT split:   0%|          | 0/10647458 [00:00<?, ? examples/s]

现在开始读/下载 Annotation

In [ ]:
!pip install progress

  Preparing metadata (setup.py) ... done
  Created wheel for progress: filename=progress-1.6-py3-none-any.whl size=9613 sha256=243d7a6c820a604a04c6eb5ea8f1c8629a20aeea07690c023869e9a3ef5207f1
  Stored in directory: /root/.cache/pip/wheels/a2/68/5f/c339b20a41659d856c93ccdce6a33095493eb82c3964aac5a1
Successfully built progress


In [ ]:
from progress.bar import Bar
text_dict = {}
for keys in ds.keys():
  print(keys)
print(len(ds['FLT']))
#print(ds['FLT'][0]['YoutubeID'])
length = len(ds['FLT'])
length = 1000000
bar = Bar('Extract Annotation', fill='#', max=length)
ind = 0
for clip in ds['FLT']:
  #print(clip['Caption'])
  text_dict[clip['YoutubeID']] = clip['Caption']
  ind = ind + 1
  bar.next()
  if ind > 1000000:
    break

# Save the text dictionary to a json file
import json
with open('InternVid_10m_text_part1.json', 'w') as file:
    json.dump(text_dict, file, indent=4)


FLT
10647458


KeyboardInterrupt: 

## WordNet取用词汇

In [ ]:
import nltk
from nltk.corpus import wordnet as wn

# Download WordNet data
nltk.download('wordnet')

# Get synsets for the word 'car'
#synsets = wn.synsets('smile')
#for synset in synsets:
#    print(synset.name(), synset.definition())

# Define a function to get words by part of speech
def get_words_by_pos(pos):
    words = set()
    for synset in wn.all_synsets(pos):
        for lemma in synset.lemmas():
            words.add(lemma.name().replace('_', ' '))
    return words

# Extract action verbs
action_verbs = get_words_by_pos(wn.VERB)
# Filter out only commonly used action verbs
common_action_verbs = {word for word in action_verbs if len(wn.synsets(word, pos=wn.VERB)) > 5}
# Extract activity nouns
activity_nouns = get_words_by_pos(wn.NOUN)

# Filter out only commonly used activity nouns
common_activity_nouns = {word for word in activity_nouns if len(wn.synsets(word, pos=wn.NOUN)) > 5}

[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
print(f"len(action_verbs): {len(action_verbs)}")
print(f"len(common_action_verbs): {len(common_action_verbs)}")
print(f"len(activity_nouns): {len(activity_nouns)}")
print(f"len(common_activity_nouns): {len(common_activity_nouns)}")
print(list(common_activity_nouns)[:50])

len(action_verbs): 586
len(common_action_verbs): 586
len(activity_nouns): 1200
len(common_activity_nouns): 1200
['row', 'values', 'board', 'Crown', 'mile', 'stuff', 'diamond', 'Day', 'finish', 'superior', 'energy', 'nut', 'grounds', 'state', 'As', 'Father', 'character', 'land', 'force', 'grain', 'proof', 'flush', 'pace', 'Harris', 'digs', 'crop', 'joint', 'green', 'kick', 'education', 'wings', 'crystal', 'men', 'Restoration', 'beat', 'nucleus', 'center', 'deal', 'origin', 'drag', 'organization', 'Ward', 'lot', 'picket', 'colours', 'mind', 'right', 'use', 'muller', 'dressing']


In [ ]:
sub_common_action_verbs = {word for word in action_verbs if len(wn.synsets(word, pos=wn.VERB)) > 3}
print(f"sub_common_action_verbs: {len(sub_common_action_verbs)}")
sub_common_activity_nouns = {word for word in activity_nouns if len(wn.synsets(word, pos=wn.NOUN)) > 3}
print(f"sub_common_activity_nouns: {len(sub_common_activity_nouns)}")

sub_common_action_verbs: 1452
sub_common_activity_nouns: 3310


In [ ]:
ind = 0
#for vb in action_verbs:
for vb in common_action_verbs:
  #print(vb)
  ind = ind + 1
  if ind > 100:
    break

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.corpus import wordnet as wn
import spacy
import json

# Load spaCy English model
nlp = spacy.load('en_core_web_sm')

# Download necessary NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Define keywords
action_verbs = list(common_action_verbs)
activity_nouns = list(common_activity_nouns)
 #["meeting", "concert", "class", "discussion", "game", "party"]
# Function to extract sentences with keywords
def extract_human_activity_sentences(text):
    sentences = sent_tokenize(text)
    human_activity_sentences = []
    for sentence in sentences:
        words = word_tokenize(sentence)
        if any(word in expanded_keywords for word in words):
            human_activity_sentences.append(sentence)
    return human_activity_sentences

# Expand keywords with synonyms using WordNet
def get_synonyms(word):
    synonyms = set()
    for syn in wn.synsets(word):
        for lemma in syn.lemmas():
            synonyms.add(lemma.name())
    return synonyms

expanded_keywords = set(action_verbs + activity_nouns)
for word in action_verbs + activity_nouns:
    expanded_keywords.update(get_synonyms(word))

# Define a list of common phrases (manual curation or additional NLP required)
common_phrases = [
    "going to", "attending a", "participating in", "working on", "engaging in",
    "spending time", "taking part in", "joining the", "involved in", "involved with",
    "on a walk", "at the gym", "during the meeting", "in the office", "with friends",
    "on a run", "at the event", "practicing for", "preparing for", "in the class"
]

# Sample text dataset
text = """
John is going for a run in the park. The dog is eating its food. Mary is reading a book.
There was a meeting at the office today. People are enjoying the concert. They plan to play a game of soccer.
"""
# Process text with spaCy
doc = nlp(text.lower())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## 判断是人物主语/主体

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.corpus import wordnet as wn
import spacy
import json

human_list = ["people", "person", "man", "woman", "child", "children", "girl", "boy"]
# Expand keywords with synonyms using WordNet
def get_synonyms(word):
    synonyms = set()
    for syn in wn.synsets(word):
        for lemma in syn.lemmas():
            synonyms.add(lemma.name().replace('_', ' '))
    return synonyms

#for wd in human_list:
#  syn_wd = get_synonyms(wd)
#  print(syn_wd)

expanded_human_list = set(human_list)
for word in human_list:
   expanded_human_list.update(get_synonyms(word))
   #print(get_synonyms(word))

print(list(expanded_human_list))

# Function to check if the subject of a sentence is human
def is_human_subject(token):
    if token.dep_ == "nsubj" or token.dep_ == "ROOT":
        if token.ent_type_ == "PERSON" or token.lemma_ in expanded_human_list:
            return True
    return False

['citizenry', 'woman', 'Man', 'cleaning woman', 'Isle of Man', 'female child', 'male child', 'fry', 'person', 'individual', 'fair sex', 'boy', 'somebody', 'tyke', 'human race', 'the great unwashed', 'small fry', 'humans', 'nestling', 'fille', 'womanhood', 'shaver', 'multitude', 'youngster', 'military personnel', 'man', 'military man', 'valet de chambre', 'world', 'mass', 'adult female', 'kid', 'girlfriend', 'nipper', 'son', 'mankind', 'little girl', 'char', 'minor', 'young woman', 'human beings', 'someone', 'hoi polloi', 'mortal', 'piece', 'humankind', 'daughter', 'human', 'tiddler', 'people', 'miss', 'gentleman', 'charwoman', 'young lady', 'girl', 'serviceman', 'adult male', 'cleaning lady', "gentleman's gentleman", 'soul', 'homo', 'tike', 'child', 'humanity', 'baby', 'children', 'lady friend', 'missy', 'human being', 'masses', 'valet']


In [ ]:
# Extract sentences with human subjects performing activities
human_activity_sentences = []

for sent in doc.sents:
    for token in sent:
        if is_human_subject(token): #and (token.lemma_ in common_action_verbs or any(phrase in sent.text for phrase in common_phrases)):
            print(token)
            human_activity_sentences.append(sent.text)
            break
print(human_activity_sentences)

john
mary
people
['\njohn is going for a run in the park.', 'mary is reading a book.\n', 'people are enjoying the concert.']


In [ ]:
# Extract sentences
activity_sentences = extract_human_activity_sentences(text)
if activity_sentences == "":
  print("no activity")
for sentence in activity_sentences:
    print(sentence)



John is going for a run in the park.
The dog is eating its food.
Mary is reading a book.
There was a meeting at the office today.
They plan to play a game of soccer.


## 批量筛选人物活动

In [ ]:
# read from InternVid_10m_text_part1.json
#text_path = "./InternVid_10m_text_part1.json"
text_path = "./InternVid_p1_text_dict.json"
#output_path = "./InternVid_10m_text_part1_human_v0.json"
output_path = "./InternVid_p1_text_dict_human_sub1_v0.json"
# Open the file in read mode
with open(text_path, 'r') as file:
    # Load the JSON data
    text_data = json.load(file)
ind = 0
human_count = 0
out_dict = {}
for k in text_data.keys():
  text = text_data[k]
  doc = nlp(text.lower())
  #print(f"{ind}: {text}")
  human_activity_sentences = []
  for sent in doc.sents:
    for token in sent:
        if is_human_subject(token): #and (token.lemma_ in common_action_verbs or any(phrase in sent.text for phrase in common_phrases)):
            #print(f"{ind}: Human:{token}, {text}")
            human_count = human_count + 1
            human_activity_sentences.append(sent.text)
            out_dict[k] = sent.text
            break

  if human_activity_sentences == []:
    #print(f"{ind}: No Activity: {text}")
    #print(text)
    pass

  #for sentence in human_activity_sentences:
    #print(sentence)
  #  pass
  ind = ind + 1
  if(ind%2000==0):
    print(f"{ind}:{human_count}")
  if ind > 20000:
    break

print(f" Human activity count: {human_count}/{ind}")
with open(output_path, 'w') as file:
    json.dump(out_dict, file, indent=4)

2000:1040
4000:2091
6000:3125
8000:4163
10000:5226
12000:6271
14000:7351
16000:8391
18000:9408
20000:10445
 Human activity count: 10445/20001


In [ ]:
# read from InternVid_10m_text_part1.json
text_path = "./InternVid_10m_text_part1.json"
# Open the file in read mode
with open(text_path, 'r') as file:
    # Load the JSON data
    text_data = json.load(file)
ind = 0
for k in text_data.keys():
  text = text_data[k]

  #print(f"{ind}: {text}")
  activity_sentences = extract_human_activity_sentences(text)
  if activity_sentences == "":
    print(f"{ind}: No Activity: {text}")
    #print(text)
  for sentence in activity_sentences:
    #print(sentence)
    pass

  ind = ind + 1
  if ind > 1000:
    break

In [ ]:
import spacy

# Load spaCy English model
nlp = spacy.load('en_core_web_sm')

# Sample sentence
sentence = "Mary Henderson is reading a book."
#sentence = "a person is putting onions and carrots into a foil pan"
# Process the sentence
doc = nlp(sentence)

# Find and print details about the token "person"
for token in doc:
    if token.text.lower() == "henderson":
        print(f"Token: {token.text}")
        print(f"  Lemma: {token.lemma_}")
        print(f"  POS: {token.pos_}")
        print(f"  Dependency: {token.dep_}")
        print(f"  Head: {token.head.text}")
        print(f"  Children: {[child.text for child in token.children]}")
        print(f"  Entity Type: {token.ent_type_}")

# Print the entire dependency parse for the sentence
print("\nDependency Parse:")
for token in doc:
    print(f"{token.text} ({token.dep_}) -> {token.head.text}")


Token: Henderson
  Lemma: Henderson
  POS: PROPN
  Dependency: nsubj
  Head: reading
  Children: ['Mary']
  Entity Type: PERSON

Dependency Parse:
Mary (compound) -> Henderson
Henderson (nsubj) -> reading
is (aux) -> reading
reading (ROOT) -> reading
a (det) -> book
book (dobj) -> reading
. (punct) -> reading
